In [1]:
pip install openneuro-py

  Using cached click-8.1.8-py3-none-any.whl (98 kB)
  Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
You should consider upgrading via the 'c:\Users\ferna\AppData\Local\Programs\Python\Python39\python.exe -m pip install --upgrade pip' command.


In [ ]:
import openneuro as on

dataset_id = 'ds006803'
download_dir = './data_openneuro'

print(f"Iniciando la descarga del dataset: {dataset_id}...")

# Descarga el dataset completo
on.download(dataset=dataset_id, target_dir=download_dir)

print(f"Descarga completada en: {download_dir}")

## Descargar solo los archivos relacionados con una sesión específica
#on.download(dataset=dataset_id, target_dir=download_dir, include=['sub-01e'])

Iniciando la descarga del dataset: ds006803...

👋 Hello! This is openneuro-py 2023.1.0. Great to see you! 🤗

   👉 Please report problems 🤯 and bugs 🪲 at
      https://github.com/hoechenberger/openneuro-py/issues

🌍 Preparing to download ds006803 …


📁 Traversing directories for ds006803 : 0 entities [00:00, ? entities/s]

📥 Retrieving up to 7 files (5 concurrent downloads). 
✅ Finished downloading ds006803.
 
🧠 Please enjoy your brains.
 
Descarga completada en: ./data_openneuro


Re-downloading .bidsignore: file size mismatch.: 0.00B [00:00, ?B/s]

Re-downloading README: file size mismatch.: 0.00B [00:00, ?B/s]

Re-downloading dataset_description.json: file size mismatch.: 0.00B [00:00, ?B/s]

Re-downloading CHANGES: file size mismatch.: 0.00B [00:00, ?B/s]

Re-downloading extra_metadata.xlsx: file size mismatch.: 0.00B [00:00, ?B/s]

Re-downloading participants.tsv: file size mismatch.: 0.00B [00:00, ?B/s]

Re-downloading participants.json: file size mismatch.: 0.00B [00:00, ?B/s]

In [11]:
pip install mne pandas

  Using cached pandas-2.3.3-cp39-cp39-win_amd64.whl (11.4 MB)
  Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\ferna\AppData\Local\Programs\Python\Python39\python.exe -m pip install --upgrade pip' command.


In [4]:
# Libraries
import pandas as pd
import numpy as np
import mne
from pathlib import Path
from scipy.signal.windows import gaussian
from mne.time_frequency import tfr_array_morlet
import gc

In [6]:
main_path = 'data_openneuro'
#Subjects lit

sub_list = pd.read_csv(f'{main_path}/participants.tsv',  sep='\t')
print(sub_list.head())



  participant_id  Gender  Age Group
0        sub-01c    Male   19   con
1        sub-02c    Male   19   con
2        sub-03c  Female   19   con
3        sub-04c    Male   20   con
4        sub-05c    Male   19   con


In [19]:

CWT_DIR = Path('D:\cwt_results_complex')
CWT_DIR.mkdir(exist_ok=True)
freqs = np.linspace(4, 12, num=12)
n_cycles = freqs /2



n = 55

for index, row in sub_list.iloc[n:].iterrows():
#for index, row in sub_list.iterrows():
    sub_id = row['participant_id']
    raw = mne.io.read_raw_eeglab(f'{main_path}/{sub_id}/ses-2/eeg/{sub_id}_ses-2_task-STEMSKILLS_eeg.set')
    df_events = pd.read_csv(f'{main_path}/{sub_id}/arithmetic_responses.csv')
    all_epochs_data = []

    fs = raw.info['sfreq']
    # Leer los nombres desde el tsv
    df_channels = pd.read_csv(f'{main_path}/{sub_id}/ses-2/eeg/{sub_id}_ses-2_electrodes.tsv', sep='\t')
    ch_pos = {row['name']: np.array([row['x'], row['y'], row['z']]) / 1000 for _, row in df_channels.iterrows()}
    montage = mne.channels.make_dig_montage(ch_pos=ch_pos, coord_frame='head')
    raw.set_montage(montage)


    samples = (df_events['Question_appearance'] * fs - 3 *fs).astype(int)
    event_id_val = 1
    quest = np.zeros((len(samples), 3), dtype=int)
    quest[:, 0] = samples

    samples = (df_events['Answer_time'] * fs - 3 *fs).astype(int)
    event_id_val = 1
    answer = np.zeros((len(samples), 3), dtype=int)
    answer[:, 0] = samples

    print(f'Procesando {sub_id}')
    for i in range(len(quest)):
        #segmentar epocas 
        t_start = quest[i, 0]/fs 
        t_end = answer[i, 0]/fs

        duracion_minima = n_cycles[0] / freqs[0] 
        if (t_end - t_start) < duracion_minima:
            continue

        segment_raw = raw.copy().crop(tmin=t_start, tmax=t_end)
        segment = segment_raw.get_data()
        segment = segment.astype(np.float32)
        
        n_channels, n_times = segment.shape
        
        #aplicar ventana gaussiana
        window = gaussian(n_times, std=n_times/4)
        segment_windowed = segment * window

        #aplicar transformada wavelets

        tfr = tfr_array_morlet(segment_windowed[np.newaxis, :], sfreq=fs, 
                            freqs=freqs, n_cycles=n_cycles, output='complex') # tfr tiene forma: (1, n_channels, n_freqs, n_times)
        
        tfr_data = tfr[0]
        for ch_idx, ch_name in enumerate(raw.ch_names):
            for f_idx, f_val in enumerate(freqs):
                row = {
                    'epoch': i,
                    'channel': ch_name,
                    'frequency': f_val,
                }

                time_values = tfr_data[ch_idx, f_idx, :]
                for t_idx, val in enumerate(time_values):
                    row[f't_{t_idx}'] = val
                all_epochs_data.append(row)

    df_results = pd.DataFrame(all_epochs_data)
    df_results.to_csv(CWT_DIR / f"{sub_id}.csv", index=False)
    print(f'Completado: {sub_id}')
    del all_epochs_data
    del raw




# Graficar
#raw.plot(events=events, event_id=event_dict, n_channels=8, start=110,duration=20, title="EEG con Marcas de Preguntas (CSV)",clipping=None)





C:\Users\ferna\AppData\Local\Temp\ipykernel_8372\266383093.py:13: RuntimeWarning: Data will be preloaded. preload=False or a string preload is not supported when the data is stored in the .set file
  raw = mne.io.read_raw_eeglab(f'{main_path}/{sub_id}/ses-2/eeg/{sub_id}_ses-2_task-STEMSKILLS_eeg.set')


Procesando sub-36e
Completado: sub-36e


C:\Users\ferna\AppData\Local\Temp\ipykernel_8372\266383093.py:13: RuntimeWarning: Data will be preloaded. preload=False or a string preload is not supported when the data is stored in the .set file
  raw = mne.io.read_raw_eeglab(f'{main_path}/{sub_id}/ses-2/eeg/{sub_id}_ses-2_task-STEMSKILLS_eeg.set')


Procesando sub-37e
Completado: sub-37e


C:\Users\ferna\AppData\Local\Temp\ipykernel_8372\266383093.py:13: RuntimeWarning: Data will be preloaded. preload=False or a string preload is not supported when the data is stored in the .set file
  raw = mne.io.read_raw_eeglab(f'{main_path}/{sub_id}/ses-2/eeg/{sub_id}_ses-2_task-STEMSKILLS_eeg.set')


...

Procesando sub-41e
Completado: sub-41e


C:\Users\ferna\AppData\Local\Temp\ipykernel_8372\266383093.py:13: RuntimeWarning: Data will be preloaded. preload=False or a string preload is not supported when the data is stored in the .set file
  raw = mne.io.read_raw_eeglab(f'{main_path}/{sub_id}/ses-2/eeg/{sub_id}_ses-2_task-STEMSKILLS_eeg.set')


Procesando sub-42e
Completado: sub-42e
Procesando sub-43e


C:\Users\ferna\AppData\Local\Temp\ipykernel_8372\266383093.py:13: RuntimeWarning: Data will be preloaded. preload=False or a string preload is not supported when the data is stored in the .set file
  raw = mne.io.read_raw_eeglab(f'{main_path}/{sub_id}/ses-2/eeg/{sub_id}_ses-2_task-STEMSKILLS_eeg.set')


Completado: sub-43e


In [ ]:
CWT_DIR = Path('D:\cwt_results')

for index, row in sub_list.iterrows:
    sub_id = row['participant_id']
    cwt = pd.read_csv(f'{CWT_DIR}/{sub_id}.csv')
    


KeyError: 0